<a href="https://colab.research.google.com/github/studtannta9166/DocuSense/blob/main/DocuSense.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
# Schritt 1: Alle Bibliotheken installieren
!pip install gradio==4.44.0 pdfplumber python-docx google-generativeai openpyxl -q
#api,pdf read,docx read

In [40]:
#Schritt 2: API Key sicher einrichten
import anthropic
import os
from google.colab import userdata #to access api saved

#Claude API Key wird hier sicher geladen
client = anthropic.Anthropic(api_key=userdata.get('GEMINI_API_KEY'))

print("Claude API erfolgreich verbunden!")

Claude API erfolgreich verbunden!


In [41]:
#verfügar Model
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gem

In [42]:
import google.generativeai as genai
from google.colab import userdata

# API Key sicher laden
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

# Modell auswählen
model = genai.GenerativeModel('gemini-2.5-flash')

# Test
response = model.generate_content("Sag nur: DocuSense API funktioniert!")
print(response.text)

DocuSense API funktioniert!


In [43]:
# Schritt 3: Dokument einlesen (PDF oder Word)
import pdfplumber
from docx import Document as DocxDocument

def dokument_lesen(datei_pfad):
    text = ""

    # Wenn es eine PDF Datei ist
    if datei_pfad.endswith('.pdf'):
        with pdfplumber.open(datei_pfad) as pdf:
            for seite in pdf.pages:
                text += seite.extract_text() or ""
        print(f"PDF eingelesen! {len(pdf.pages)} Seite(n) gefunden.")

    # Wenn es eine Word Datei ist
    elif datei_pfad.endswith('.docx'):
        doc = DocxDocument(datei_pfad)
        for absatz in doc.paragraphs:
            text += absatz.text + "\n"
        print("Word Dokument eingelesen!")

    else:
        print("Dateiformat nicht unterstützt! Nur PDF oder DOCX.")
        return None

    return text

print("Parser bereit!")

Parser bereit!


In [44]:
# Schritt 4: Dokument mit Gemini analysieren
def dokument_analysieren(text):

    prompt = f"""
    Du bist ein Experte für Dokumentenanalyse.
    Analysiere dieses Dokument und antworte auf Deutsch in diesem Format:

    DOKUMENTTYP:
    (Was ist das? z.B. Rechnung, Vertrag, Compliance-Bericht, Brief)

    WICHTIGE INFORMATIONEN:
    (Liste die wichtigsten Felder: Datum, Namen, Beträge, Fristen usw.)

    ZUSAMMENFASSUNG:
    (2-3 Sätze — was ist der Inhalt des Dokuments?)

    AUFFÄLLIGKEITEN:
    (Gibt es fehlende Informationen oder etwas Ungewöhnliches?)

    Hier ist der Dokumenttext:
    {text}
    """

    antwort = model.generate_content(prompt)
    return antwort.text

print("Analyse-Funktion bereit!")

Analyse-Funktion bereit!


In [45]:
# Schritt 5: Benutzeroberfläche mit Gradio
import gradio as gr

def komplett_analysieren(datei):
    if datei is None:
        return "❌ Bitte eine Datei hochladen!"

    # Schritt 1: Text aus Datei lesen
    text = dokument_lesen(datei.name)

    if text is None:
        return "❌ Datei konnte nicht gelesen werden!"

    if len(text.strip()) == 0:
        return "❌ Dokument ist leer!"

    # Schritt 2: KI analysiert den Text
    ergebnis = dokument_analysieren(text)

    return ergebnis

# Interface bauen
interface = gr.Interface(
    fn=komplett_analysieren,
    inputs=gr.File(label="📄 Dokument hochladen (PDF oder DOCX)"),
    outputs=gr.Textbox(label="🧠 KI Analyse", lines=20),
    title="DocuSense 🔍",
    description="Lade ein Dokument hoch — die KI analysiert es sofort!",
)

interface.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/analytics.py:93: UserWarning: IMPORTANT: You are using gradio version 4.44.0, however version 4.44.1 is available, please upgrade. 
--------
  await asyncio.wait_for(


AttributeError: module 'gradio' has no attribute 'Request'